In [ ]:
!pip install -U pageindex openai python-dotenv -q

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

In [ ]:
from pageindex import PageIndexClient
from openai    import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("Clients initialized")

In [ ]:
PDF_PATH = ".downloads/introml.pdf"   

result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded | doc_id: {doc_id}")

In [ ]:
print("⏳ Indexing in progress...")

while True:
    status = pi_client.get_document(doc_id).get("status")
    print(f"   Status: {status}")
    if status == "completed":
        print("Tree index ready")
        break
    elif status == "failed":
        print("Indexing failed")
        break
    time.sleep(5)

In [ ]:
tree_result    = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")

In [ ]:
def print_tree(nodes, indent=0):
    for node in nodes:
        pad  = "  " * indent + ("└─ " if indent else "")
        page = node.get("page_index", "?")
        print(f"{pad}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print_tree(pageindex_tree)

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

print(f"Total nodes: {count_nodes(pageindex_tree)}")

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core retrieval: LLM reasons over the document tree
    and returns node IDs most likely to answer the query.
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:120],
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a precise document retrieval assistant.
Given a query and a document tree (hierarchical table of contents with summaries),
identify the node IDs that contain the most relevant information to answer the query.
Reason step by step over the tree structure before deciding.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Respond ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
query  = "What are the main conclusions of the document?"

result = llm_tree_search(query, pageindex_tree)

print(f"Query: {query}")
print(f"\nReasoning:\n{result.get('thinking', '')}")
print(f"\nSelected Nodes: {result.get('node_list', [])}")

In [ ]:
def find_nodes_by_ids(tree: list, node_ids: list) -> list:
    """Recursively find and return node objects matching given IDs."""
    found = []
    for node in tree:
        if node["node_id"] in node_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], node_ids))
    return found


def fetch_node_content(doc_id: str, node_ids: list) -> list:
    """
    Fetch full section content from PageIndex API for given node IDs.
    Falls back to tree-level text if API fetch fails.
    """
    contents = []
    for nid in node_ids:
        try:
            result = pi_client.get_node(doc_id, nid)
            contents.append({
                "node_id": nid,
                "title":   result.get("title", ""),
                "page":    result.get("page_index", "?"),
                "content": result.get("text", ""),
            })
        except Exception as e:
            print(f"⚠️  Could not fetch node {nid}: {e}")
    return contents

In [ ]:
def generate_answer(query: str, node_contents: list, model: str = "gpt-4o") -> str:
    """
    Generate a grounded, cited answer from retrieved node contents.
    """
    if not node_contents:
        return "No relevant content found in the document."

    context_parts = []
    for nc in node_contents:
        context_parts.append(
            f"[Section: {nc['title']} | Page {nc['page']}]\n{nc['content']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a precise document Q&A assistant.
Answer the question using ONLY the provided document sections.
Always cite the section title and page number in your answer.
If the context does not contain the answer, say "The document does not cover this."

Question: {query}

Document Sections:
{context}

Answer:"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [ ]:
def vectorless_rag(query: str, tree: list, doc_id: str) -> str:
    """
    Full end-to-end Vectorless RAG pipeline:
    1. LLM Tree Search  → identify relevant node IDs
    2. Fetch Content    → retrieve full section text
    3. Generate Answer  → grounded, cited response
    """
    print(f"🔍 Query: {query}")

    search_result = llm_tree_search(query, tree)
    node_ids      = search_result.get("node_list", [])
    print(f"🎯 Nodes selected: {node_ids}")

    node_contents = fetch_node_content(doc_id, node_ids)

    answer = generate_answer(query, node_contents)
    return answer

In [ ]:
query  = "Summarize the key findings of this document."
answer = vectorless_rag(query, pageindex_tree, doc_id)

print("\n📄 Answer:")
print(answer)

In [ ]:
EXPERT_RULES = """
Domain-specific retrieval guidelines:
- For questions about methodology or process: prioritize sections with 'Method', 'Approach', 'Framework'
- For questions about results or findings: prioritize sections with 'Results', 'Findings', 'Outcomes', 'Analysis'
- For questions about recommendations: prioritize 'Conclusion', 'Recommendations', 'Future Work'
- For technical questions: look for 'Architecture', 'Implementation', 'Technical', 'System'
- Always prefer specific sections over introductory/overview sections when both are present
"""

def llm_tree_search_with_expert(
    query: str, tree: list, expert_rules: str, model: str = "gpt-4o"
) -> dict:
    """Tree search enhanced with domain expert guidance."""

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:120],
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert and document retrieval specialist.
Use the expert rules below to guide your retrieval decisions.

Expert Rules:
{expert_rules}

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Respond ONLY in this exact JSON format:
{{
  "thinking": "<your expert reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
query = "What are the key recommendations?"

print("── Basic Tree Search ──")
basic = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print("\n── Expert-Guided Tree Search ──")
guided = llm_tree_search_with_expert(query, pageindex_tree, EXPERT_RULES)
print("Nodes:    ", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [ ]:
def expert_vectorless_rag(query: str, tree: list, doc_id: str, rules: str) -> str:
    """Expert-guided end-to-end vectorless RAG pipeline."""
    print(f"🔍 Query: {query}")

    search_result = llm_tree_search_with_expert(query, tree, rules)
    node_ids      = search_result.get("node_list", [])
    print(f"🎯 Nodes: {node_ids}")

    node_contents = fetch_node_content(doc_id, node_ids)
    return generate_answer(query, node_contents)


answer = expert_vectorless_rag(
    query="What methodology was used in this document?",
    tree=pageindex_tree,
    doc_id=doc_id,
    rules=EXPERT_RULES,
)
print("\n📄 Expert Answer:")
print(answer)

In [ ]:
conversation = []

def chat_with_document(question: str, doc_id: str) -> str:
    """Multi-turn document conversation using PageIndex Chat API."""
    global conversation

    conversation.append({"role": "user", "content": question})

    response       = pi_client.chat_completions(messages=conversation, doc_id=doc_id)
    assistant_msg  = response["choices"][0]["message"]["content"]

    conversation.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg

In [ ]:
questions = [
    "Give me a brief overview of this document.",
    "What is the most important section?",
    "Summarize the conclusions.",
]

for q in questions:
    print(f"👤 {q}")
    reply = chat_with_document(q, doc_id)
    print(f"🤖 {reply[:350]}...")
    print("-" * 60)

In [ ]:
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt -q

In [ ]:
openai_key = os.getenv("OPENAI_API_KEY")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print(".env ready for self-hosted mode")

In [ ]:
PDF_PATH = ".downloads/introml.pdf"  

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
TREE_JSON = "/path/to/your_document_pageindex.json"   # ← change this

with open(TREE_JSON) as f:
    local_tree = json.load(f)

print(f"Local tree: {count_nodes(local_tree)} nodes")
print_tree(local_tree)

query  = "What are the key findings?"
result = llm_tree_search(query, local_tree)
nodes  = find_nodes_by_ids(local_tree, result.get("node_list", []))
answer = generate_answer(query, nodes)
print("\nAnswer:", answer)